# 08 — Grad-CAM: model nhìn vào đâu?

**Chủ: Phong Trần** · lý thuyết: `docs/LY_THUYET.md` 6.6 · code: `src/gtsrb/explain/gradcam.py`

## Công thức (Selvaraju et al. 2017)

```
alpha_k^c = (1/Z) · Σ_i Σ_j  ∂y^c / ∂A^k_ij          ← trọng số của kênh k
L^c       = ReLU( Σ_k alpha_k^c · A^k )               ← bản đồ nhiệt
```

**Trực giác từng bước:**
1. `∂y^c/∂A^k` nói "nếu kênh k mạnh lên thì điểm số lớp c tăng hay giảm".
2. Gộp theo không gian cho **một** con số mỗi kênh: *"kênh này quan trọng cỡ nào cho lớp c"*.
3. Tổ hợp các kênh theo trọng số đó.
4. `ReLU` giữ **chỉ** phần **hỗ trợ** lớp c (phần âm là bằng chứng **chống lại** lớp c).

**Vì sao dùng feature map cuối:** nó có ngữ nghĩa cao nhất **mà vẫn còn giữ vị trí**
(các lớp fully-connected sau đó mới xoá hết vị trí).

Chạy `make gradcam` để sinh toàn bộ lưới hình.

In [ ]:
# Cho notebook thấy src/.  Chạy ô này đầu tiên.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)          # mọi đường dẫn trong repo là tương đối từ gốc

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from gtsrb import CLASS_NAMES, NUM_CLASSES
print("torch", torch.__version__, "| MPS", torch.backends.mps.is_available(),
      "| CUDA", torch.cuda.is_available())

In [ ]:
# Tiện ích dùng chung: tìm và nạp lại một run đã train
import glob, json
sys.path.insert(0, str(ROOT / "scripts"))
from evaluate import load_run, test_loader_for
from gtsrb.utils.seed import pick_device

device = pick_device("auto")

def find_runs(prefix=""):
    """Danh sách thư mục run có checkpoint, lọc theo tiền tố tên model."""
    dirs = [pathlib.Path(p) for p in sorted(glob.glob("artifacts/runs/*"))]
    return [d for d in dirs if (d / "best.pt").exists() and d.name.startswith(prefix)]

def latest(prefix):
    runs = find_runs(prefix)
    if not runs:
        raise FileNotFoundError(
            f"Chưa có run nào tên bắt đầu bằng {prefix!r}. Chạy: make train-{prefix[:2]}")
    return runs[-1]

print("Các run đã có:")
for d in find_runs(): print("  ", d.name)

## 1. ★ Phải báo cáo ĐỘ PHÂN GIẢI heatmap trước khi nói gì khác

In [ ]:
from gtsrb.explain.gradcam import GradCAM, feature_map_size

rows = []
for run in find_runs():
    model, cfg, _ = load_run(run, device)
    h, w = feature_map_size(model, cfg.data.img_size, device)
    rows.append({
        "model": model.model_name,
        "input": f"{cfg.data.img_size}x{cfg.data.img_size}",
        "target layer": type(model.gradcam_target_layer).__name__,
        "feature map": f"{h}x{w}",
        "1 ô heatmap ~": f"{cfg.data.img_size/h:.0f}x{cfg.data.img_size/w:.0f} pixel",
    })
    del model
print(pd.DataFrame(rows).to_string(index=False))

**★ Đây là GIỚI HẠN quan trọng nhất, phải nói ra trước khi diễn giải bất cứ hình nào.**

M2 ở input 48×48 cho feature map cuối **3×3**. Upsample lên 48×48 nghĩa là mỗi "ô"
heatmap tương ứng **16×16 pixel** — một phần ba chiều rộng ảnh. Nói *"heatmap đúng vào
biển báo"* ở độ phân giải đó là một phát biểu **rất lỏng**: cả ảnh chỉ có 9 ô.

M3 ở input 224 cho **7×7** → nét hơn hẳn. Nhưng đó là lợi thế về **độ phân giải**,
**không** phải vì model "hiểu" hơn. Trộn hai thứ đó là một lỗi lập luận.

Muốn nhìn rõ hơn ở M2 thì lấy `stages[2]` (6×6), nhưng ngữ nghĩa thấp hơn — đánh đổi.

## 2. Nhóm ảnh ĐÚNG

In [ ]:
from gtsrb.data.dataset import GTSRBDataset
from gtsrb.data.transforms import build_normalize

def make_dataset(cfg):
    return GTSRBDataset(
        index_csv=cfg.data.index_csv, split="test",
        img_size=cfg.data.img_size, preprocess=cfg.data.preprocess,
        normalize=cfg.data.normalize,
        mean=cfg.data.get("mean"), std=cfg.data.get("std"),
        processed_dir=cfg.data.get("processed_dir", "data/processed"),
        cache_size=cfg.data.get("cache_size"), return_uint8=True)

def gradcam_panels(run, want_correct=True, n=6, scan=3000):
    """Trả danh sách panel (ảnh, overlay, caption) cho dự đoán đúng hoặc SAI."""
    model, cfg, _ = load_run(run, device)
    ds = make_dataset(cfg)
    norm = build_normalize(ds.mean, ds.std)
    panels = []
    cam = GradCAM(model)
    try:
        step = max(1, len(ds) // scan)
        for i in range(0, len(ds), step):
            if len(panels) >= n: break
            t, label = ds[i]
            img = t.permute(1, 2, 0).numpy()
            x = norm(t).unsqueeze(0).to(device)
            with torch.no_grad():
                p = torch.softmax(model(x).float(), 1)[0]
            pred = int(p.argmax()); conf = float(p[pred])
            if (pred == label) != want_correct: continue
            hm = cam(x, class_idx=pred)
            panels.append({"img": img, "overlay": GradCAM.overlay(img, hm),
                           "cap": f"thật: {CLASS_NAMES[label][:17]}\n"
                                  f"đoán: {CLASS_NAMES[pred][:17]} ({conf:.2f})"})
    finally:
        cam.close()      # BẮT BUỘC tháo hook
    del model
    return panels, cfg

def show_panels(panels, title):
    if not panels: print("không có panel:", title); return
    fig, axes = plt.subplots(2, len(panels), figsize=(2.3 * len(panels), 5.4), squeeze=False)
    for c, pn in enumerate(panels):
        axes[0][c].imshow(pn["img"]); axes[0][c].set_title(pn["cap"], fontsize=7.5)
        axes[1][c].imshow(pn["overlay"])
        for r in (0, 1): axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
    axes[0][0].set_ylabel("ảnh gốc", fontsize=9); axes[1][0].set_ylabel("Grad-CAM", fontsize=9)
    fig.suptitle(title, fontsize=11); plt.tight_layout(); plt.show()

run = find_runs()[0]
panels, cfg = gradcam_panels(run, want_correct=True, n=6)
show_panels(panels, f"{run.name} — dự đoán ĐÚNG")

## 3. ★ Nhóm ảnh SAI — thú vị hơn nhóm đúng

Heatmap trên ảnh đúng gần như luôn nằm trên biển báo — **không nói được gì mới**.
Ảnh SAI mới cho biết model đang nhìn vào đâu khi nó lầm.

In [ ]:
panels, cfg = gradcam_panels(run, want_correct=False, n=6)
show_panels(panels, f"{run.name} — dự đoán SAI: model nhìn vào đâu?")

**Cách phân tích ca sai — viết ít nhất 3 ca vào báo cáo:**

Với mỗi ảnh sai, trả lời 2 câu:
1. Heatmap có nằm **trên biển báo** không?
   - **Có** → model nhìn đúng chỗ mà vẫn suy luận sai → vấn đề ở **đặc trưng**, không ở
     **định vị**. Thường là hai lớp quá giống nhau (xem `notebooks/05`: nhóm biển
     "HẾT hiệu lực" 6/41/42, hoặc biển giới hạn tốc độ 3→5).
   - **Không** → model bị nền/viền ảnh làm nhiễu → augmentation hoặc cắt ROI có thể giúp.
2. Lớp bị đoán sai có nằm trong bảng "cặp nhầm nhiều nhất" ở `notebooks/05` không?
   Nếu có thì đây là lỗi **hệ thống**, không phải ngẫu nhiên.

**★ Giới hạn phải nêu:** Grad-CAM chỉ cho biết **ở đâu**, không cho biết **đặc trưng gì**,
và **không phải bằng chứng nhân quả** — model có thể nhìn đúng chỗ mà vẫn suy luận sai.

## 4. Attention có TRÔI khi có nhiễu?

In [ ]:
from gtsrb.robustness.corruptions import apply_corruption

model, cfg, _ = load_run(run, device)
ds = make_dataset(cfg); norm = build_normalize(ds.mean, ds.std)
t, label = ds[7]
base = t.permute(1, 2, 0).numpy()
rng = np.random.default_rng(0)

cam = GradCAM(model)
panels = []
try:
    for kind, sev in [("clean", 0), ("fog", 3), ("low_light", 3),
                      ("motion_blur", 3), ("gauss_noise", 3), ("occlusion", 3)]:
        arr = base if kind == "clean" else apply_corruption(base, kind, sev, rng)
        tt = torch.from_numpy(np.ascontiguousarray(arr)).permute(2, 0, 1)
        x = norm(tt).unsqueeze(0).to(device)
        with torch.no_grad():
            p = torch.softmax(model(x).float(), 1)[0]
        pred = int(p.argmax())
        hm = cam(x, class_idx=pred)
        mark = "OK" if pred == label else "SAI"
        panels.append({"img": arr, "overlay": GradCAM.overlay(arr, hm),
                       "cap": f"{kind}\n{mark}: {CLASS_NAMES[pred][:15]} ({float(p[pred]):.2f})"})
finally:
    cam.close()
show_panels(panels, f"Attention có TRÔI khi có nhiễu? (nhãn thật: {CLASS_NAMES[label]})")
del model

**Liên hệ với `notebooks/07`.** Nếu attention **trôi khỏi** biển báo khi thêm nhiễu
và đúng lúc đó dự đoán sai, thì ta có một **giải thích cơ chế** cho con số robustness —
không chỉ nói "accuracy giảm" mà nói được **giảm vì sao**. Đó là chất lượng phân tích
mà giảng viên tìm kiếm.